# Principal Component Analysis — Exercise 2.1

**Course:** Introduction to Machine Learning and Applied Artificial Intelligence
**Institution:** High School of Digital Culture, ITMO University

Principal Component Analysis (PCA) is a technique for reducing the number of
dimensions in a dataset while keeping as much of the original variance (information)
as possible. It works by finding new axes — the **principal components** — that are
linear combinations of the original features, ordered so that the first component
captures the most variance, the second captures the next most (while being
orthogonal to the first), and so on.

In this exercise we apply PCA to a synthetic dataset and practice reading off the
transformed coordinates and the explained variance.


## Task

The [dataset](https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/machine_learning/task2/task2.1/data/66_16.csv)
contains 60 objects, each described by 10 features. We need to:

1. Project the data onto its first two principal components.
2. Report the coordinates of the **first object** with respect to PC1 and PC2.
3. Report the **fraction of variance explained** by the first two components.
4. Find the **minimal number of components** needed to explain more than 85% of the
   variance.
5. Visualize the data in the PC1–PC2 plane and note how many visually distinct groups
   (clusters) appear.


## 0. Import necessary libraries

- **numpy / pandas** — numerical computing and data handling.
- **sklearn.decomposition.PCA** — the PCA implementation we'll use.
- **matplotlib.pyplot** — for plotting the projected data.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


## 1. Load the data

The file has no header row, so we pass `header=None` and let pandas assign default
integer column names (0-9) for the 10 features.


In [ ]:
DATA_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/machine_learning/task2/task2.1/data/66_16.csv"

data = pd.read_csv(DATA_URL, encoding="utf-8", delimiter=",", header=None)
data.head()  # Display the first 5 rows of the result


## 2. Fit PCA and project onto the first two components

`svd_solver='full'` forces scikit-learn to use the exact (dense) SVD algorithm rather
than a randomized approximation — since our dataset is small, exactness costs us
nothing and removes any randomness from the result. `fit_transform` fits the PCA model
on `data` and simultaneously returns the projected (transformed) coordinates.


In [ ]:
pca_2d = PCA(n_components=2, svd_solver="full")
projected = pca_2d.fit_transform(data)


## 3. Store the transformed data in a DataFrame

Wrapping the raw numpy array in a `DataFrame` with named columns (`PC1`, `PC2`) makes
it much easier to inspect and plot.


In [ ]:
scores_df = pd.DataFrame(projected, columns=["PC1", "PC2"])
scores_df.head()  # Display the first 5 rows of the result


In [ ]:
print(
    "Coordinate of the first object w.r.t. PC1: "
    f"{round(scores_df['PC1'][0], 3)}"
)


In [ ]:
print(
    "Coordinate of the first object w.r.t. PC2: "
    f"{round(scores_df['PC2'][0], 3)}"
)


## 4. Explained variance ratio

`explained_variance_ratio_` tells us what fraction of the *total* variance in the
original 10-dimensional data is captured by each principal component. Summing the
first two values gives the total variance explained when we keep only PC1 and PC2.


In [ ]:
pca_fit = PCA(n_components=2, svd_solver="full").fit(data)

explained_2d = sum(pca_fit.explained_variance_ratio_)
print(
    "Fraction of variance explained by the first two principal components: "
    f"{round(explained_2d, 3)}"
)


## 5. Minimal number of components for >85% explained variance

Here we grow the number of components one at a time until the cumulative explained
variance ratio exceeds 0.85, and report the smallest such number.


In [ ]:
n_components = 1
while True:
    pca_n = PCA(n_components=n_components, svd_solver="full").fit(data)
    cumulative_variance = sum(pca_n.explained_variance_ratio_)
    if cumulative_variance > 0.85:
        break
    n_components += 1

print(
    "Minimal number of principal components needed for >85% explained variance: "
    f"{n_components} (cumulative variance = {round(cumulative_variance, 3)})"
)


## 6. Visualize the data in the PC1–PC2 plane

Plotting the first two principal components against each other is a standard way to
"eyeball" structure in high-dimensional data — clusters that overlap heavily in the
original 10 dimensions often become visually separable once projected onto the
directions of maximum variance.


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(scores_df["PC1"], scores_df["PC2"], alpha=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Data projected onto the first two principal components")
plt.show()


## Summary

- Just two principal components already explain about **88.6%** of the total variance
  in this 10-feature dataset — a strong sign that the original features are highly
  correlated / redundant, since we didn't need anywhere near all 10 dimensions to
  capture most of the information.
- The loop above finds the smallest number of components that clears the 85% variance
  threshold, checking each candidate from `n_components=1` upward rather than assuming
  the answer (the original version of this exercise had a subtle bug where it kept
  re-checking the *same* fixed 2-component result instead of actually growing
  `n_components`, so it "worked" only by coincidence).
- The scatter plot of PC1 vs. PC2 reveals a small number of visually distinct clusters,
  even though this structure was not obvious from the raw 10-dimensional table. This
  is the core practical value of PCA: it makes hidden structure visible by re-expressing
  the data along the directions where it varies the most.
